# 04 · Ансамбль, пороги, фінальні сабміти (CPU)
**Вхід:** змагання, вихід ноутбука 00 (`split.csv`) і файли `*_val.npy` / `*_test.npy` трьох прогонів ноутбука 02: XLM-R (малий train), XLM-R (великий train), mDeBERTa (великий train). **Вихід:** `submission_blend.csv` (ансамбль) і `submission_best_single.csv` (найкраща одиночна модель). Час ~1 хв.

Ансамбль це зважене середнє ймовірностей + множники порогів для класів, які підбираються на валідації за macro-F1. Щоб перевірити, що ми не підігнали все під валідацію, спершу підбір іде на половині A, а перевірка на половині B.

Результат: на валідації ансамбль 0.9353 проти 0.9301 у найкращої одиночної моделі. Public leaderboard: ансамбль 0.93864, одиночна mDeBERTa 0.93714. Ці два файли обрані як final.

## Шапка: дані та спліт
Читає дані змагання та `split.csv` з ноутбука 00. Спліт груповий: людський текст і його AI/fusion версії завжди в одній частині, тож валідація (8 005 текстів) чесно імітує тест.

In [1]:
import os, glob, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.metrics import f1_score, classification_report, confusion_matrix

ID, TEXT, LABEL = "id", "text", "label"
SEED = 42
LABELS = ["human", "ai", "fusion"]
OUT = "/kaggle/working"

def find(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
ss = pd.read_csv(find("sample_submission.csv"))

# спліт береться з ноутбука 0 (split.csv): вся група "людський текст + його AI/fusion версії" лежить в одній частині
sp = pd.read_csv(find("split.csv"))
d = train.merge(sp, on=ID)
tr = d[d.part == "tr"].reset_index(drop=True)
va = d[d.part == "va"].reset_index(drop=True)
l2i = {l: i for i, l in enumerate(LABELS)}
ytr, yva = tr[LABEL].map(l2i).values, va[LABEL].map(l2i).values
print(train.columns.tolist(), "| train:", len(tr), "val:", len(va), "test:", len(test))

def save_sub(p, fn):  # p = матриця ймовірностей для test, пише файл у форматі Kaggle
    out = pd.DataFrame({ID: test[ID], LABEL: np.array(LABELS)[p.argmax(1)]})
    out.set_index(ID).loc[ss[ID]].reset_index().to_csv(f"{OUT}/{fn}", index=False)

['id', 'text', 'lang', 'domain', 'label'] | train: 100002 val: 8005 test: 25000


## Завантаження ймовірностей
Кожен прогін енкодера зберіг ймовірності для валідації й тесту (`*_val.npy`, `*_test.npy`), тут вони підтягуються автоматично з усіх входів.

In [2]:
P = {}
for p in glob.glob("/kaggle/input/**/*_val.npy", recursive=True):
    P[os.path.basename(p)[:-8]] = (np.load(p), np.load(p[:-8] + "_test.npy"))
names_m = list(P); V = [P[n][0] for n in names_m]; T = [P[n][1] for n in names_m]
f1p = lambda p, idx: f1_score(yva[idx], p[idx].argmax(1), average="macro")
ALL = np.ones(len(yva), bool)
for n, v in zip(names_m, V):
    print(n, round(f1p(v, ALL), 4))

enc_mdeberta-v3-base_ht_512_dom_100k 0.9301
enc_xlm-roberta-base_ht_512_dom 0.904
enc_xlm-roberta-base_ht_512_dom_100k 0.9221


## Пошук ваг ансамблю і порогів
- **Ваги:** внесок кожної моделі в середнє.
- **Біаси:** множники для ймовірностей класів, щоб підняти macro-F1 (наприклад, частіше казати fusion).
- Спершу підбираємо на половині A, перевіряємо на B, щоб побачити, чи не підігнались під валідацію. Це корисно розповісти на захисті.

In [3]:
rng = np.random.default_rng(SEED)
mix = lambda arrs, w: sum(wi * a for wi, a in zip(w, arrs))

def search(idx):
    best = (-1, None, None)
    for _ in range(300):
        w = rng.dirichlet(np.ones(len(V)))
        p = mix(V, w)
        for b1 in [0.8, 1, 1.2, 1.5]:
            for b2 in [0.8, 1, 1.2, 1.5, 2]:
                bias = np.array([1, b1, b2])          # human, ai, fusion
                f = f1p(p * bias, idx)
                if f > best[0]:
                    best = (f, w, bias)
    return best

A = np.arange(len(yva)) % 2 == 0
fA, wA, bA = search(A)
print("підбір на половині A:", round(fA, 4), "| перевірка на половині B:", round(f1p(mix(V, wA) * bA, ~A), 4))

f, w, bias = search(ALL)
print("фінал (вся валідація):", round(f, 4), "\nваги:", dict(zip(names_m, w.round(2))), "\nбіаси:", bias)

підбір на половині A: 0.9379 | перевірка на половині B: 0.9304
фінал (вся валідація): 0.9353 
ваги: {'enc_mdeberta-v3-base_ht_512_dom_100k': np.float64(0.55), 'enc_xlm-roberta-base_ht_512_dom': np.float64(0.42), 'enc_xlm-roberta-base_ht_512_dom_100k': np.float64(0.03)} 
біаси: [1.  0.8 0.8]


## Фінальні файли
`submission_blend.csv` це ансамбль із порогами, `submission_best_single.csv` це безпечний варіант (найкраща одна модель). Kaggle дозволяє обрати 2 фінальні сабміти.

In [4]:
save_sub(mix(T, w) * bias, "submission_blend.csv")
bi = int(np.argmax([f1p(v, ALL) for v in V]))
save_sub(T[bi], "submission_best_single.csv")
print("найкраща одиночна модель:", names_m[bi])

найкраща одиночна модель: enc_mdeberta-v3-base_ht_512_dom_100k


In [5]:
pb = mix(V, w) * bias
print(classification_report(yva, pb.argmax(1), target_names=LABELS, digits=3))

              precision    recall  f1-score   support

       human      0.897     0.887     0.892      1950
          ai      0.995     0.997     0.996      3556
      fusion      0.916     0.921     0.918      2499

    accuracy                          0.946      8005
   macro avg      0.936     0.935     0.935      8005
weighted avg      0.946     0.946     0.946      8005

